In [1]:
import os
import sys
import json
import glob
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from pathlib import Path
from torch.optim import AdamW
import torch.nn.functional as F
from IPython.display import display
from sklearn.metrics import f1_score
import torch.nn.functional as functional
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    matthews_corrcoef,
    cohen_kappa_score,
    log_loss,
    roc_auc_score,
    average_precision_score
)

/Users/lorenaandravacarean/Desktop/Fake News/Model/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/lorenaandravacarean/Desktop/Fake News/Model/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "roberta-base"
LABELS = ["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]
LABEL2ID = {lbl: i for i, lbl in enumerate(LABELS)}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

out_dir = Path("eval_results")
out_dir.mkdir(parents=True, exist_ok=True)
summary_csv = out_dir / "model_eval_summary.csv"
pd.set_option("display.precision", 4)

In [3]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [4]:
fever_claims_path = os.path.join(os.getcwd(), "Dataset", "fever")
fever_test_path = os.path.join(fever_claims_path, "test.json")

In [5]:
def load_fever_claims(path_):
    data_list = []
    df = pd.read_json(path_, lines=True)

    for idx, row in df.iterrows():
        claim_text = row["claim"]
        evidence_text = row.get("evidence_text", "")
        label_str = row["label"]
        label_id = LABEL2ID.get(label_str)
        data_list.append({
            "claim": claim_text,
            "evidence": evidence_text,
            "label_id": label_id
        })
    
    return data_list

In [6]:
class FeverDataset(Dataset):

    def __init__(self, data_list, tokenizer, max_len=256):
        self.data_list = data_list
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data_list)
    
    def __getitem__(self, idx):
        sample = self.data_list[idx]
        claim_text = sample["claim"]
        evidence_text = sample["evidence"]
        label_id = sample["label_id"]

        encoded = self.tokenizer(
            text=claim_text,
            text_pair=evidence_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        input_ids = encoded["input_ids"].squeeze(0)
        attn_mask = encoded["attention_mask"].squeeze(0)

        return {
            "input_ids": input_ids,
            "attention_mask": attn_mask,
            "label": torch.tensor(label_id, dtype=torch.long)
        }

In [7]:
class BiGRUCNNClassifier(nn.Module):

    def __init__(self, roberta_name="roberta-base", hidden_size=128, num_labels=3):
        super().__init__()
        # STEP 1: load RoBERTa as the encoder
        self.roberta = AutoModel.from_pretrained(roberta_name)

        # STEP 2: BiGRU
        # roberta output: 768 dimensional embeddings
        self.gru = nn.GRU(
            input_size=768,
            hidden_size=hidden_size,
            batch_first=True,
            bidirectional=True
        )

        # STEP 3: CNN
        self.conv1 = nn.Conv1d(
            in_channels=2*hidden_size,
            out_channels=hidden_size,
            kernel_size=3,
            padding=1
        )

        # STEP 4: Max-pooling
        self.pool = nn.AdaptiveMaxPool1d(1)

        # STEP 5: Fully Connected Layer
        self.classifier = nn.Linear(128, num_labels)


    def forward(self, input_ids, attention_mask):
        # STEP 1: RoBERTa embeddings
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        token_embeds = outputs.last_hidden_state

        # STEP 2: BiGRU -> [B, T, 2*hidden_size]
        gru_out, _ = self.gru(token_embeds)

        # STEP 3: CNN -> [B, 128, T]
        x = gru_out.permute(0, 2, 1) # din [B, T, 2*hidden_size] in [B, 2*hidden_size, T]
        x = torch.relu(self.conv1(x))
        x = self.pool(x).squeeze(2) # [B, 128]

        # STEP 4: FC
        logits = self.classifier(x)
        return logits

In [8]:
def evaluate(model, loader, loss_fn, class_names=None, compute_auc=False):
    # loss, acc, macro f1 on val loader
    model.eval()
    all_logits = []
    all_preds = []
    all_labels = []
    total_loss = 0.0
    n_batches = 0
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(logits, labels)

            total_loss += loss.item()
            n_batches += 1

            probs = F.softmax(logits, dim=-1)  # for log_loss / AUCs
            preds = probs.argmax(dim=-1)

            all_logits.append(probs.detach().cpu().numpy())
            all_preds.append(preds.detach().cpu().numpy())
            all_labels.append(labels.detach().cpu().numpy())

    y_prob = np.concatenate(all_logits, axis=0)   # [N, C]
    y_pred = np.concatenate(all_preds, axis=0)    # [N]
    y_true = np.concatenate(all_labels, axis=0)   # [N]
    num_classes = y_prob.shape[1]

    avg_loss = total_loss / max(n_batches, 1)
            
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)

    # Macro/weighted P/R/F1 + per-class
    precision_w, recall_w, f1_w, support = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    precision_m, recall_m, f1_m, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    precision_none, recall_none, f1_none, support_none = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    report_str = classification_report(
        y_true, y_pred, target_names=class_names if class_names else None, zero_division=0
    )

    # Agreement / correlation style metrics
    mcc = matthews_corrcoef(y_true, y_pred) if num_classes > 1 else 0.0
    kappa = cohen_kappa_score(y_true, y_pred)

    # Prob-based scores
    ll = log_loss(y_true, y_prob, labels=list(range(num_classes)))

    metrics = {
        "loss": avg_loss,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "precision_macro": precision_m,
        "recall_macro": recall_m,
        "f1_macro": f1_m,
        "precision_weighted": precision_w,
        "recall_weighted": recall_w,   # <- your requested recall
        "f1_weighted": f1_w,
        "mcc": mcc,
        "cohen_kappa": kappa,
        "log_loss": ll,
        "per_class": {}
    }

    # Per-class detail
    for c in range(num_classes):
        cname = class_names[c] if class_names and c < len(class_names) else str(c)
        metrics["per_class"][cname] = {
            "precision": precision_none[c],
            "recall": recall_none[c],
            "f1": f1_none[c],
            "support": int(support_none[c]),
            "tp": int(cm[c, c]),
            "fp": int(cm[:, c].sum() - cm[c, c]),
            "fn": int(cm[c, :].sum() - cm[c, c]),
            "tn": int(cm.sum() - (cm[:, c].sum() + cm[c, :].sum() - cm[c, c]))
        }

    # Optional AUCs (one-vs-rest, macro average)
    if compute_auc and num_classes > 1:
        try:
            roc_auc_macro = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
            pr_auc_macro = average_precision_score(
                np.eye(num_classes)[y_true], y_prob, average="macro"
            )
            metrics["roc_auc_macro_ovr"] = roc_auc_macro
            metrics["pr_auc_macro_ovr"] = pr_auc_macro
        except Exception as e:
            metrics["roc_auc_macro_ovr"] = None
            metrics["pr_auc_macro_ovr"] = None
            metrics["auc_error"] = str(e)

    return metrics, cm, report_str


In [9]:
model = BiGRUCNNClassifier().to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
final_models_path = os.path.join(os.getcwd(), "claim+evidence")
best_lr = 3e-5 
batch_size = 32

optimizer = torch.optim.AdamW(model.parameters(), lr=best_lr, weight_decay=1e-2)
    
test_data = load_fever_claims(fever_test_path)
test_dataset = FeverDataset(test_data, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

loss_fn = torch.nn.CrossEntropyLoss()

In [11]:
def _overall_row(name, m):
    return {
        "model": name,
        "loss": m.get("loss"),
        "accuracy": m.get("accuracy"),
        "balanced_accuracy": m.get("balanced_accuracy"),
        "precision_macro": m.get("precision_macro"),
        "recall_macro": m.get("recall_macro"),
        "f1_macro": m.get("f1_macro"),
        "precision_weighted": m.get("precision_weighted"),
        "recall_weighted": m.get("recall_weighted"),
        "f1_weighted": m.get("f1_weighted"),
        "mcc": m.get("mcc"),
        "cohen_kappa": m.get("cohen_kappa"),
        "log_loss": m.get("log_loss"),
        "roc_auc_macro_ovr": m.get("roc_auc_macro_ovr"),
        "pr_auc_macro_ovr": m.get("pr_auc_macro_ovr"),
    }

In [12]:
class_names = ["SUPPORTS", "NOT ENOUGH INFO", "REFUTES"]


if not summary_csv.exists():
    pd.DataFrame(columns=list(_overall_row("model", {}))).to_csv(summary_csv, index=False)


for model_path in glob.glob(os.path.join(final_models_path, "*.pth")):
    name_model = os.path.basename(model_path).replace(".pth", "")
    print(f"\nEvaluating model: {os.path.basename(model_path)}")

    model = BiGRUCNNClassifier(roberta_name=MODEL_NAME, hidden_size=128, num_labels=3)
    state = torch.load(model_path, map_location="cpu")
    model.load_state_dict(state)
    model.to(device)

    metrics, cm, report_str = evaluate(
        model, test_loader, loss_fn,
        class_names=class_names, compute_auc=False
    )

    overall_df = pd.DataFrame([_overall_row(name_model, metrics)])
    display(overall_df)

    per_class_df = pd.DataFrame(metrics["per_class"]).T[
        ["precision","recall","f1","support","tp","fp","fn","tn"]
    ]
    display(per_class_df)

    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    print("\nConfusion Matrix:")
    display(cm_df)

    print("\nClassification report:\n")
    print(report_str)

    sys.stdout.flush()

    pd.DataFrame([_overall_row(name_model, metrics)]).to_csv(summary_csv, mode="a", header=False, index=False)

    per_class_df.to_csv(out_dir / f"{name_model}_per_class.csv", index=True)
    cm_df.to_csv(out_dir / f"{name_model}_confusion_matrix.csv")
    with open(out_dir / f"{name_model}_classification_report.txt", "w") as f:
        f.write(report_str)
    with open(out_dir / f"{name_model}_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    # Free memory before next checkpoint
    del model, state
    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        elif torch.backends.mps.is_available():
            torch.mps.empty_cache()
    except Exception:
        pass


Evaluating model: stage2_epoch4_valf10.9566.pth


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


,model,loss,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,mcc,cohen_kappa,log_loss,roc_auc_macro_ovr,pr_auc_macro_ovr
0,stage2_epoch4_valf10.9566,0.1381,0.9575,0.9557,0.9575,0.9557,0.9564,0.9578,0.9575,0.9574,0.9363,0.9361,0.1383,None,None


,precision,recall,f1,support,tp,fp,fn,tn
SUPPORTS,0.9239,0.9568,0.9401,4443.0,4251.0,350.0,192.0,7962.0
NOT ENOUGH INFO,0.9486,0.9104,0.9291,3895.0,3546.0,192.0,349.0,8668.0
REFUTES,1.0000,0.9998,0.9999,4417.0,4416.0,0.0,1.0,8338.0



Confusion Matrix:


...

,model,loss,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,mcc,cohen_kappa,log_loss,roc_auc_macro_ovr,pr_auc_macro_ovr
0,stage2_epoch8_valf10.9572,0.2312,0.9561,0.9534,0.958,0.9534,0.9547,0.9575,0.9561,0.9559,0.9348,0.9339,0.2311,None,None


,precision,recall,f1,support,tp,fp,fn,tn
SUPPORTS,0.9080,0.9725,0.9391,4443.0,4321.0,438.0,122.0,7874.0
NOT ENOUGH INFO,0.9659,0.8875,0.9251,3895.0,3457.0,122.0,438.0,8738.0
REFUTES,1.0000,1.0000,1.0000,4417.0,4417.0,0.0,0.0,8338.0



Confusion Matrix:


,SUPPORTS,NOT ENOUGH INFO,REFUTES
SUPPORTS,4321,122,0
NOT ENOUGH INFO,438,3457,0
REFUTES,0,0,4417



Classification report:

                 precision    recall  f1-score   support

       SUPPORTS       0.91      0.97      0.94      4443
NOT ENOUGH INFO       0.97      0.89      0.93      3895
        REFUTES       1.00      1.00      1.00      4417

       accuracy                           0.96     12755
      macro avg       0.96      0.95      0.95     12755
   weighted avg       0.96      0.96      0.96     12755

